In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from hydra import compose, initialize

from main.model.script.hydra_beans import KdConfig

config_name = "train-local.yaml"
with initialize(version_base=None, config_path="../../../conf/"):
    cfg: KdConfig = compose(config_name="train-local.yaml")

To evaluate the good contribution of modalities we see MRR of it with and without the metric <br>
So if I have EEG, Aud, Txt and Vid and decide to ablate *vid* I measure:

A:MRR_mean modalities model that can use Vid but without video in inputstream <br>
B:MRR_mean across modalities of the ablated model

If B > A → Video is likely hurting other modalities<br>
If B < A → Video is likely helping them<br>
If B ~ A → Video has little effect on them<br>

In [ ]:
from main.model.neegavi.factory import Factory
from main.model.neegavi.utils import get_model_ckpt

# TODO change?
baseline_checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/epochepoch=45-stepstep=117484.ckpt"
ckpt = get_model_ckpt(baseline_checkpoint_path)
baseline = Factory.best_inference().build()
baseline.load_state_dict(ckpt, strict=False)
baseline.eval()

In [ ]:
import lightning
from main.model.neegavi.train_utils import KdTrainDataModule
from main.model.neegavi.training import EasyEegAviKdVateMaskedModule

trainer = lightning.Trainer(precision="16-mixed")

# Audio

In [ ]:
from main.core_data.media.audio import Audio

audio_less_checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-aud/2026-03-22_22-14-48/checkpoints/epochepoch=38-stepstep=99567.ckpt"

inference_ckpt = get_model_ckpt(audio_less_checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set([Audio.modality_code()])).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

full_datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[baseline.pivot.code] + baseline.fusion_keys()
)

baseline_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=full_datamodule)
baseline_audioless_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=datamodule, )
audio_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )

baseline_results = trainer.validate(baseline_module, datamodule=full_datamodule)
baseline_audioless_results = trainer.validate(baseline_audioless_module, datamodule=datamodule)
audio_less_results = trainer.validate(audio_less_module, datamodule=datamodule)

Main effect of audio: maybe positive, neutral, or slightly negative

# Txt

In [ ]:
from main.core_data.media.text import Text

checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-txt/2026-03-22_11-24-33/checkpoints/epochepoch=38-stepstep=99567.ckpt"

inference_ckpt = get_model_ckpt(checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set([Text.modality_code()])).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

baseline_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=full_datamodule)
baseline_modless_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=datamodule, )
audio_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )

baseline_results = trainer.validate(baseline_module, datamodule=full_datamodule)
baseline_modless_results = trainer.validate(baseline_modless_module, datamodule=datamodule)
audio_less_results = trainer.validate(audio_less_module, datamodule=datamodule)

Text seems to fluctuate a lot from seed to seed. <br>
The modality is unstable.

> What if the increase in performance by removing audio is because it mitigates the presence of txt and thus removing modaltiies while txt is in it always proves gains?

# ECG

In [ ]:
from main.core_data.media.text import Text

audio_less_checkpoint_path = ""

inference_ckpt = get_model_ckpt(audio_less_checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set(Text.modality_code())).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

full_datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[baseline.pivot.code] + baseline.fusion_keys()
)

baseline_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=full_datamodule)
baseline_modless_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=datamodule, )
audio_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )

baseline_results = trainer.validate(baseline_module, datamodule=full_datamodule)
baseline_modless_results = trainer.validate(baseline_modless_module, datamodule=datamodule)
audio_less_results = trainer.validate(audio_less_module, datamodule=datamodule)

In [ ]:
baseline_results

In [ ]:
audio_less_results

# MoCo less

In [1]:
from hydra import compose, initialize
import lightning
from main.model.neegavi.train_utils import KdTrainDataModule
from main.model.neegavi.training import EasyEegAviKdVateMaskedModule

from main.model.script.hydra_beans import KdConfig
from main.model.neegavi.factory import Factory
from main.model.neegavi.utils import get_model_ckpt

config_name = "train-local.yaml"
with initialize(version_base=None, config_path="../../../conf/"):
    cfg: KdConfig = compose(config_name="train-local.yaml")

trainer = lightning.Trainer(precision="16-mixed")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'train-local.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Baseline model load

In [2]:
mrr_check_keys = [
    'val/fused/mrr_ecg',
    'val/fused/mrr_aud',
    'val/fused/mrr_vid',
    # 'val/fused/mrr_txt', -> We are ablating text
    'val/fused/mrr_eeg',
    'val/fused/mrr_mean',
]

In [3]:
baselines = []
seed_ckpt = [
    # Seed=42. For some reason this are broken. (Model architecture mismatch on keys?)
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-moco-best/2026-03-19_15-32-23/checkpoints/last.ckpt",
    # Seed=150
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-moco-best/2026-03-24_00-32-18/checkpoints/epochepoch=38-stepstep=99567.ckpt"
]
for path in seed_ckpt:
    ckpt = get_model_ckpt(path)
    baseline = Factory.best_inference().build()
    # Load state of the seed ckpt
    baseline.load_state_dict(ckpt, strict=False)
    baseline.eval()
    # Append the built model
    baselines.append(baseline)

Ablated txt model load

In [4]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[baselines[0].pivot.code] + baselines[0].fusion_keys()
)

In [5]:
baseline_mod_less_results = []
for b in baselines:
    baseline_mod_less_module = EasyEegAviKdVateMaskedModule(b, None, datamodule=datamodule, )
    baseline_mod_less_results.append(trainer.validate(baseline_mod_less_module, datamodule=datamodule)[0])

You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.09319126605987549      │
│    val/fused/alignment_ecg    │     -0.07486465573310852      │
│    val/fused/alignment_eeg    │      0.1790904998779297       │
│    val/fused/alignment_txt    │     -0.06974675506353378      │
│    val/fused/alignment_vid    │      0.12579084932804108      │
│     val/fused/margin_aud      │      0.2772243618965149       │
│     val/fused/margin_ecg      │      0.16955387592315674      │
│     val/fused/margin_eeg      │      0.3371985852718353       │
│     val/fused/margin_txt      │      0.09508460015058517      │
│     val/fused/margin_vid      │      0.39437222480773926      │
│ val/fused/meanR@1-3-5-10_aud  │      0.8843636512756348       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.21155959367752075      │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9860917329788208       │
│ val/fused/meanR@1-3-5-10_mean │      0.5818352699279785       │
│ val/fused/meanR@1-3-5-10_txt  │     0.029801076278090477      │
│ val/fused/meanR@1-3-5-10_vid  │      0.7973601818084717       │
│       val/fused/mrr_aud       │      0.8599337935447693       │
│       val/fused/mrr_ecg       │      0.1713300198316574       │
│       val/fused/mrr_eeg       │      0.9800578951835632       │
│      val/fused/mrr_mean       │      0.5491713881492615       │
│       val/fused/mrr_txt       │     0.023113396018743515      │
│       val/fused/mrr_vid       │      0.7114218473434448       │
│      val/fused/top10_aud      │      0.9304202795028687       │
│      val/fused/top10_ecg      │      0.33141130208969116      │
│      val/fused/top10_eeg      │      0.9949050545692444       │
│     val/fused/top10_mean      │      0.6453568339347839       │
│      val/fused/top10_txt      │      0.04845946654677391      │
│      val/fused/top10_vid      │      0.9215880036354065       │
│      val/fused/top1_aud       │      0.8211023807525635       │
│      val/fused/top1_ecg       │      0.0936298742890358       │
│      val/fused/top1_eeg       │       0.970745325088501       │
│      val/fused/top1_mean      │      0.4964584410190582       │
│      val/fused/top1_txt       │     0.010619204491376877      │
│      val/fused/top1_vid       │      0.5861955285072327       │
│      val/fused/top3_aud       │      0.8826127052307129       │
│      val/fused/top3_ecg       │      0.18478858470916748      │
│      val/fused/top3_eeg       │      0.9875913858413696       │
│      val/fused/top3_mean      │      0.5775818228721619       │
│      val/fused/top3_txt       │     0.024827998131513596      │
│      val/fused/top3_vid       │      0.8080882430076599       │
│      val/fused/top5_aud       │      0.9033191800117493       │
│      val/fused/top5_ecg       │      0.23640857636928558      │
│      val/fused/top5_eeg       │      0.9911249876022339       │
│      val/fused/top5_mean      │      0.6079438328742981       │
│      val/fused/top5_txt       │     0.035297635942697525      │
│      val/fused/top5_vid       │      0.8735688924789429       │
│        val/fusion-loss        │      3.7288026809692383       │
│        val/fusion/aud         │       3.478576898574829       │
│        val/fusion/ecg         │       5.91384220123291        │
│        val/fusion/eeg         │       2.42000412940979        │
│        val/fusion/txt         │       6.042628288269043       │
│        val/fusion/vid         │      3.1774497032165527       │
│           val/loss            │      3.7288026809692383       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.1320195198059082       │
│    val/fused/alignment_ecg    │      0.2077609896659851       │
│    val/fused/alignment_eeg    │      0.1400308459997177       │
│    val/fused/alignment_txt    │      0.08619217574596405      │
│    val/fused/alignment_vid    │      0.14835308492183685      │
│     val/fused/margin_aud      │      0.27854952216148376      │
│     val/fused/margin_ecg      │      0.4161883592605591       │
│     val/fused/margin_eeg      │      0.28491073846817017      │
│     val/fused/margin_txt      │      0.3013633191585541       │
│     val/fused/margin_vid      │      0.3650020360946655       │
│ val/fused/meanR@1-3-5-10_aud  │      0.9567981958389282       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8625068664550781       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9617470502853394       │
│ val/fused/meanR@1-3-5-10_mean │      0.8180058598518372       │
│ val/fused/meanR@1-3-5-10_txt  │       0.546066403388977       │
│ val/fused/meanR@1-3-5-10_vid  │       0.762910783290863       │
│       val/fused/mrr_aud       │      0.9350941777229309       │
│       val/fused/mrr_ecg       │      0.7973181009292603       │
│       val/fused/mrr_eeg       │      0.9460970163345337       │
│      val/fused/mrr_mean       │      0.7744465470314026       │
│       val/fused/mrr_txt       │      0.5198565721511841       │
│       val/fused/mrr_vid       │      0.6738667488098145       │
│      val/fused/top10_aud      │      0.9876675009727478       │
│      val/fused/top10_ecg      │      0.9577155709266663       │
│      val/fused/top10_eeg      │       0.984961748123169       │
│     val/fused/top10_mean      │      0.8838645815849304       │
│      val/fused/top10_txt      │      0.5883936882019043       │
│      val/fused/top10_vid      │      0.9005847573280334       │
│      val/fused/top1_aud       │      0.9027101397514343       │
│      val/fused/top1_ecg       │      0.7031850814819336       │
│      val/fused/top1_eeg       │      0.9234941005706787       │
│      val/fused/top1_mean      │      0.7102217078208923       │
│      val/fused/top1_txt       │      0.4754711389541626       │
│      val/fused/top1_vid       │      0.5462482571601868       │
│      val/fused/top3_aud       │      0.9619367122650146       │
│      val/fused/top3_ecg       │      0.8704009056091309       │
│      val/fused/top3_eeg       │      0.9641712307929993       │
│      val/fused/top3_mean      │      0.8228382468223572       │
│      val/fused/top3_txt       │      0.5516003370285034       │
│      val/fused/top3_vid       │      0.7660818696022034       │
│      val/fused/top5_aud       │      0.9748782515525818       │
│      val/fused/top5_ecg       │      0.9187259674072266       │
│      val/fused/top5_eeg       │      0.9743610620498657       │
│      val/fused/top5_mean      │      0.8550988435745239       │
│      val/fused/top5_txt       │      0.5688005089759827       │
│      val/fused/top5_vid       │      0.8387282490730286       │
│        val/fusion-loss        │       2.943612575531006       │
│        val/fusion/aud         │      3.0037081241607666       │
│        val/fusion/ecg         │      2.1381113529205322       │
│        val/fusion/eeg         │      2.9312548637390137       │
│        val/fusion/txt         │       3.455925941467285       │
│        val/fusion/vid         │       2.921487331390381       │
│           val/loss            │       2.943612575531006       │
└───────────────────────────────┴───────────────────────────────┘

In [9]:
import numpy as np

# I only care for MRR + meanR@
print("Baseline variance calculations")

# Without text mean
baseline_mod_less_results[0]["val/fused/mrr_mean"] = 0.8213

baseline_mod_less_results[0]["val/fused/mrr_ecg"] = 0.6902
baseline_mod_less_results[0]["val/fused/mrr_aud"] = 0.9383
baseline_mod_less_results[0]["val/fused/mrr_eeg"] = 0.9641
baseline_mod_less_results[0]["val/fused/mrr_vid"] = 0.6925
baseline_mod_less_results[0]["val/fused/mrr_txt"] = 0.5746

baseline_mod_less_results[0]["val/fused/mrr_mean"] = (
        (baseline_mod_less_results[0]["val/fused/mrr_vid"] + baseline_mod_less_results[0]["val/fused/mrr_aud"] +
         baseline_mod_less_results[0]["val/fused/mrr_ecg"] + baseline_mod_less_results[0]["val/fused/mrr_eeg"]) / 4
)
baseline_mod_less_results[1]["val/fused/mrr_mean"] = (
        (baseline_mod_less_results[1]["val/fused/mrr_vid"] + baseline_mod_less_results[1]["val/fused/mrr_aud"] +
         baseline_mod_less_results[1]["val/fused/mrr_ecg"] + baseline_mod_less_results[1]["val/fused/mrr_eeg"]) / 4
)

baseline_metrics = {}
for key in mrr_check_keys:
    values = []
    for res in baseline_mod_less_results:
        values.append(res[key])

    arr = np.array(values)
    baseline_metrics[key] = np.mean(arr)
    print(f"For key:{key} on {len(baseline_mod_less_results)} baselines mean: {np.mean(arr)} std: {np.std(arr)}")

Baseline variance calculations
For key:val/fused/mrr_ecg on 2 baselines mean: 0.7437590504646301 std: 0.05355905046463011
For key:val/fused/mrr_aud on 2 baselines mean: 0.9366970888614654 std: 0.0016029111385345574
For key:val/fused/mrr_vid on 2 baselines mean: 0.6831833744049072 std: 0.009316625595092776
For key:val/fused/mrr_eeg on 2 baselines mean: 0.9550985081672668 std: 0.009001491832733133
For key:val/fused/mrr_mean on 2 baselines mean: 0.8296845054745674 std: 0.008409505474567425


In [7]:
txt_ablate_ckpt_150 = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-txt/2026-03-25_12-03-51/checkpoints/epochepoch=38-stepstep=99567.ckpt"

inference_ckpt = get_model_ckpt(txt_ablate_ckpt_150)
mod_less_model = Factory.best_inference(disabled_supports={'txt'}).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

EegInterAviModel(
  (pivot): ModalityStream(
    (adapter): EegAdapter(
      (ff): Sequential(
        (0): Linear(in_features=3800, out_features=384, bias=True)
        (1): GELU(approximate='none')
        (2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (supports): ModuleList(
    (0-1): 2 x ModalityStream(
      (adapter): PerceiverResamplerAdapter(
        (linear_reshape): Linear(in_features=768, out_features=384, bias=True)
        (resampler): PerceiverResampler(
          (blocks): ModuleList(
            (0-1): 2 x ModuleList(
              (0): PerceiverAttention(
                (norm_latents): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (norm_media): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
                (to_q): Linear(in_features=384, out_features=768, bias=False)
                (to_k): Linear(in_features=384, out_features=768, bias=False)
                (to_v): Linear(in_features=384, out_features=

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

In [11]:
mod_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )
mod_less_results = trainer.validate(mod_less_module, datamodule=datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [10]:
# I only care for MRR + meanR@
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_metrics[key])

For seed=150
ablated_res for key: val/fused/mrr_ecg  is: 0.8040436506271362
key: val/fused/mrr_ecg  has gain of: 0.06028460016250614
ablated_res for key: val/fused/mrr_aud  is: 0.9584898948669434
key: val/fused/mrr_aud  has gain of: 0.02179280600547795
ablated_res for key: val/fused/mrr_vid  is: 0.700972855091095
key: val/fused/mrr_vid  has gain of: 0.017789480686187797
ablated_res for key: val/fused/mrr_eeg  is: 0.959257185459137
key: val/fused/mrr_eeg  has gain of: 0.004158677291870139
ablated_res for key: val/fused/mrr_mean  is: 0.8556908965110779
key: val/fused/mrr_mean  has gain of: 0.02600639103651048


In [ ]:
# Compare baseline withou txt

In [17]:
baseline_mod_less_module = EasyEegAviKdVateMaskedModule(baselines[1], None, datamodule=datamodule, )
res = trainer.validate(baseline_mod_less_module, datamodule=datamodule)[0]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.14956499636173248      │
│    val/fused/alignment_ecg    │      0.21219755709171295      │
│    val/fused/alignment_eeg    │      0.14407804608345032      │
│    val/fused/alignment_vid    │      0.15417495369911194      │
│     val/fused/margin_aud      │      0.31456637382507324      │
│     val/fused/margin_ecg      │      0.4251786768436432       │
│     val/fused/margin_eeg      │      0.2969921827316284       │
│     val/fused/margin_vid      │      0.3919265866279602       │
│ val/fused/meanR@1-3-5-10_aud  │      0.9636876583099365       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8565348982810974       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9635343551635742       │
│ val/fused/meanR@1-3-5-10_mean │      0.8902395367622375       │
│ val/fused/meanR@1-3-5-10_vid  │      0.7772011756896973       │
│       val/fused/mrr_aud       │      0.9446651935577393       │
│       val/fused/mrr_ecg       │      0.7911146879196167       │
│       val/fused/mrr_eeg       │      0.9483951330184937       │
│      val/fused/mrr_mean       │      0.8433588743209839       │
│       val/fused/mrr_vid       │      0.6892603039741516       │
│      val/fused/top10_aud      │      0.9911693334579468       │
│      val/fused/top10_ecg      │      0.9541460871696472       │
│      val/fused/top10_eeg      │       0.985947847366333       │
│     val/fused/top10_mean      │       0.96016526222229        │
│      val/fused/top10_vid      │      0.9093979001045227       │
│      val/fused/top1_aud       │      0.9165651798248291       │
│      val/fused/top1_ecg       │      0.6954969763755798       │
│      val/fused/top1_eeg       │      0.9257128238677979       │
│      val/fused/top1_mean      │      0.7753505706787109       │
│      val/fused/top1_vid       │      0.5636273622512817       │
│      val/fused/top3_aud       │      0.9666565656661987       │
│      val/fused/top3_ecg       │      0.8640856742858887       │
│      val/fused/top3_eeg       │      0.9667186737060547       │
│      val/fused/top3_mean      │      0.8948392868041992       │
│      val/fused/top3_vid       │      0.7818960547447205       │
│      val/fused/top5_aud       │      0.9803593754768372       │
│      val/fused/top5_ecg       │      0.9124107956886292       │
│      val/fused/top5_eeg       │      0.9757580161094666       │
│      val/fused/top5_mean      │      0.9306029081344604       │
│      val/fused/top5_vid       │      0.8538835048675537       │
│        val/fusion-loss        │      2.7206764221191406       │
│        val/fusion/aud         │       2.697739601135254       │
│        val/fusion/ecg         │       2.087677001953125       │
│        val/fusion/eeg         │       2.872695207595825       │
│        val/fusion/vid         │      2.8334057331085205       │
│           val/loss            │      2.7206764221191406       │
└───────────────────────────────┴───────────────────────────────┘

In [18]:
# I only care for MRR + meanR@
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - res[key])
    print()

For seed=150
ablated_res for key: val/fused/mrr_ecg  is: 0.8040436506271362
key: val/fused/mrr_ecg  has gain of: 0.012928962707519531

ablated_res for key: val/fused/mrr_aud  is: 0.9584898948669434
key: val/fused/mrr_aud  has gain of: 0.013824701309204102

ablated_res for key: val/fused/mrr_vid  is: 0.700972855091095
key: val/fused/mrr_vid  has gain of: 0.01171255111694336

ablated_res for key: val/fused/mrr_eeg  is: 0.959257185459137
key: val/fused/mrr_eeg  has gain of: 0.01086205244064331

ablated_res for key: val/fused/mrr_mean  is: 0.8556908965110779
key: val/fused/mrr_mean  has gain of: 0.012332022190093994



In [ ]:
# The full model is fairly robust to missing text at inference
# the main effect is probably not catastrophic text dependence
# training without text gives a small but consistent alignment advantage

> The model appears robust to the absence of text at inference. Training without text still yields a modest improvement on shared-modality retrieval, suggesting that text may slightly complicate alignment during training rather than being strictly required at test time.

> Although removing text slightly improved shared-modality retrieval, the gain was modest. Since the broader goal of the model is to learn richer multimodal representations, text was retained as a support modality despite this small alignment cost.

or

> In this setting, text was derived from speech rather than being an independent source of information. Given its small negative effect on shared-modality retrieval and the likely redundancy with audio, it was excluded from the final configuration.